In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import os
from utils.geo_utils import gps_to_meters  # ваш модуль с функцией gps_to_meters

def main():
    # 1. Создание папки output
    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)

    # 2. Чтение данных (без заголовков)
    file_path = "data/test_50kmh_2.xlsx"
    df = pd.read_excel(file_path, header=None, names=["time", "coord", "frame"])

    # 3. Преобразование времени в datetime и сортировка
    df["time"] = pd.to_datetime(df["time"], format="%d/%m/%Y %H:%M:%S")
    df = df.sort_values("time").reset_index(drop=True)

    # 4. Извлечение широты и долготы из строки "lat, lon"
    coords = df["coord"].str.split(",", expand=True)
    df["lat"] = coords[0].astype(float)
    df["lon"] = coords[1].astype(float)

    # 5. Опорная точка (первая после сортировки)
    ref_lat = df.loc[0, "lat"]
    ref_lon = df.loc[0, "lon"]
    print(f"Опорная точка: широта {ref_lat}, долгота {ref_lon}")

    # 6. Пересчёт в метры для каждой точки
    xs, zs = [], []
    for _, row in df.iterrows():
        x, z = gps_to_meters(row["lat"], row["lon"], ref_lat, ref_lon)
        xs.append(x)
        zs.append(z)
    df["x_m"] = xs
    df["z_m"] = zs

    # 7. Сохранение результата в папку output
    output_excel = os.path.join(output_dir, "gps_data_with_meters.xlsx")
    df.to_excel(output_excel, index=False)
    print(f"Результат сохранён в {output_excel}")

    # 8. Построение и сохранение траектории
    plt.figure(figsize=(10, 6))
    plt.plot(df["x_m"], df["z_m"], marker="o", linestyle="-", markersize=3, linewidth=1)
    plt.xlabel("X (метры, восток)")
    plt.ylabel("Z (метры, север)")
    plt.title("Траектория движения в метровой системе координат")
    plt.grid(True)
    plt.axis("equal")  # одинаковый масштаб осей

    # Сохраняем график
    output_plot = os.path.join(output_dir, "trajectory.png")
    plt.savefig(output_plot, dpi=300, bbox_inches="tight")
    print(f"График сохранён в {output_plot}")

    # Отображаем график (опционально)
    plt.show()

if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'data/test_50kmh_2.xlsx'